## **Observatorio de Accidentalidad Vial en Colombia: Integración de Datos para la Toma de Decisiones**

**Jeferson Cardona — Daniela Baena**

Introducción a la Analítica de Negocios -
Universidad de Antioquia

## **Problemática**

Los siniestros viales en Colombia son una de las principales causas de muerte en el
país, un problema que no ha disminuido pese a los esfuerzos institucionales. Solo en
2024 se registraron 8.479 muertes por siniestros viales, una tasa de 16,09
fallecidos por cada 100.000 habitantes, cifra que se mantiene en un nivel
históricamente alto tras el repunte que siguió a la pandemia (Instituto Nacional de
Medicina Legal y Ciencias Forenses [INMLCF], 2026).

En Colombia, como respuesta a esta problemática, se cuenta con el Observatorio
Nacional de Seguridad Vial (ONSV), administrado por la Agencia Nacional de
Seguridad Vial (ANSV), que se encarga de recopilar, analizar y divulgar información
sobre los siniestros viales que ocurren en el país (Agencia Nacional de Seguridad
Vial [ANSV], s. f.). Pese a que existe una fuente oficial de información, esta
suele presentar estadísticas descriptivas de una sola fuente a la vez, sin integrar
variables de otras entidades (población, contexto económico y laboral) que
permitan explicar por qué la siniestralidad se concentra en determinados
territorios o perfiles de víctima.

Esta es la necesidad de información que busca cubrir el presente observatorio: no
duplicar las estadísticas ya publicadas, sino construir, a partir de la integración
de varias fuentes, indicadores y relaciones que apoyen decisiones de priorización  de intervenciones de seguridad vial
por parte de entidades públicas y privadas.

## **Propósito estratégico**

Identificar los factores territoriales, sociodemográficos y socioeconómicos
asociados a la frecuencia de siniestros viales en los principales departamentos de
Colombia, mediante la integración de fuentes oficiales dispersas, con el fin de
crear indicadores que apoyen la priorización de intervenciones de seguridad vial
por parte de entidades públicas y privadas.

## **Preguntas a resolver**


*   **Q1 — Territorial (base de la cadena)**
¿En qué departamentos se concentra la mayor frecuencia de siniestros viales, y qué
patrones horarios y de causalidad asociados al medio de transporte ayudan a
explicar por qué unos territorios son más propensos que otros?
*   **Q2 — Sociodemográfica**
¿Cómo varía la frecuencia y gravedad de los siniestros viales según el género,
escolaridad y el grupo de edad de las personas involucradas?



*   **Q3 — Socioeconómica**
¿Los departamentos con mayor tasa de desocupación y menor actividad económica
concentran también una mayor frecuencia de siniestros de motociclistas, o esta
relación no se sostiene al comparar el conjunto de departamentos?




## **Periodicidad del análisis**


*   ANSV/ONSV — Muertes y Lesiones por eventos de transporte:actualización anual;
  histórico disponible desde 2009 para fallecidos y 2015 para lesionados.
*   DANE — Mercado laboral por departamentos (GEIH): actualización anual, con series
  históricas disponibles por departamento.



## Introducción a las bases de datos

Para este observatorio se integran **tres fuentes oficiales**, cada una con un nivel
de detalle distinto:

| Fuente | Entidad | Grano del dato | Qué aporta |
|---|---|---|---|
| **Muertes por eventos de transporte** | INMLCF (vía datos.gov.co) | Un registro por víctima fatal | Variables sociodemográficas y del hecho para cada persona fallecida |
| **Lesiones por eventos de transporte** | INMLCF (vía datos.gov.co) | Un registro por víctima lesionada | Las mismas variables, para personas que resultaron lesionadas (no fallecidas) |
| **Mercado laboral por departamentos (GEIH)** | DANE | Un registro por departamento-año | Indicadores de contexto económico (ocupación, desocupación, subocupación, población) |

Las dos primeras fuentes comparten exactamente el mismo esquema de columnas, lo que
permite **unirlas por filas** (apilarlas) en una sola tabla de víctimas. La tercera
fuente tiene un grano distinto (departamento y año, no persona), por lo que se
integra más adelante mediante un **cruce por llave** (`departamento`, `año`) para
agregarle a cada víctima el contexto laboral de su departamento en el año del hecho.

In [1]:
import requests
import pandas as pd

## Sección 1: Muertes por eventos de transporte

**Fuente:** [Muertes por eventos de transporte — Colombia](https://www.datos.gov.co/Justicia-y-Derecho/Muertes-por-eventos-de-transporte-Colombia-a-os-20/s65h-7665/about_data)
(Instituto Nacional de Medicina Legal y Ciencias Forenses, vía datos.gov.co)

Cada fila de esta base es **una persona fallecida** en un evento de transporte
(accidente de tránsito), con variables sociodemográficas y del hecho.

In [2]:
url_muertes = "https://www.datos.gov.co/resource/s65h-7665.json?$limit=73403"
respuesta = requests.get(url_muertes)
datos_muertes = respuesta.json()

df_muertes = pd.DataFrame(datos_muertes)
print(df_muertes.shape)

(73403, 38)


Le pedimos al servidor de datos.gov.co los 73,403 registros de este dataset y los convertimos en una tabla de pandas.

In [3]:
print(list(df_muertes.columns))

['id', 'a_o_del_hecho', 'grupo_de_edad_quinquenal', 'grupo_mayor_menor_de_edad', 'grupo_de_edad_judicial', 'ciclo_vital', 'sexo_de_la_victima', 'estado_civil', 'pais_de_nacimiento', 'escolaridad', 'pertenencia_grupal', 'pertenencia_etnica', 'mes_del_hecho', 'dia_del_hecho', 'rango_de_hora_del_hecho_x_3_horas', 'codigo_dane_municipio', 'municipio_del_hecho_dane', 'departamento_del_hecho_dane', 'codigo_dane_departamento', 'escenario_del_hecho', 'zona_del_hecho', 'actividad_durante_el_hecho', 'circunstancia_del_hecho_detallada', 'manera_de_muerte', 'mecanismo_causal_de_la_lesion_fatal', 'diagnostico_topografico_de_la_lesion_fatal', 'condicion_de_la_victima_at', 'medio_de_desplazamiento_o_transporte', 'servicio_del_vehiculo', 'clase_o_tipo_de_accidente_de_transporte', 'objeto_de_colision', 'servicio_del_objeto_de_colision', 'localidad_del_hecho', 'ancestro_racial', 'pueblo_indigena', 'orientacion_sexual', 'identidad_de_genero', 'transgenero']


Antes de seleccionar variables, revisamos la columna `departamento_del_hecho_dane`
en busca de inconsistencias de escritura (tildes faltantes, variantes del nombre de
Bogotá, etc.), ya que más adelante esta columna será la llave para cruzar con las
otras dos fuentes.

In [4]:
print(df_muertes["departamento_del_hecho_dane"].nunique())
print(df_muertes["departamento_del_hecho_dane"].unique())

36
['Meta' 'Cesar' 'Antioquia' 'Arauca' 'Cauca' 'Quindio' 'Santander'
 'Atlántico' 'Magdalena' 'Bolívar' 'Valle del Cauca' 'Cundinamarca'
 'Córdoba' 'Norte de Santander' 'Boyacá' 'Tolima' 'Caquetá' 'La Guajira'
 'Huila' 'Caldas' 'Nariño' 'Putumayo' 'Casanare' 'Risaralda' 'Vichada'
 'Chocó' 'Bogotá D.C.' 'Sucre' 'Sin información' 'Amazonas' 'Guaviare'
 'Archipiélago de San Andrés, Providencia y Santa Catalina' 'Bogotá, D.C.'
 'Quindío' 'Guainía' 'Vaupés']


Encontramos dos inconsistencias: `"Quindio"` (sin tilde) y `"Bogotá, D.C."` (con
coma). Las corregimos para que coincidan con la forma estándar usada en las demás
fuentes.

In [5]:
df_muertes["departamento_del_hecho_dane"] = df_muertes["departamento_del_hecho_dane"].replace({
    "Quindio": "Quindío",
    "Bogotá, D.C.": "Bogotá D.C."
})

Contamos cuántos departamentos reales tiene la base, excluyendo los registros
marcados como `"Sin información"`.

In [6]:
departamentos = df_muertes[
    df_muertes["departamento_del_hecho_dane"] != "Sin información"
]["departamento_del_hecho_dane"].unique()

print(len(departamentos))
print(departamentos)

33
['Meta' 'Cesar' 'Antioquia' 'Arauca' 'Cauca' 'Quindío' 'Santander'
 'Atlántico' 'Magdalena' 'Bolívar' 'Valle del Cauca' 'Cundinamarca'
 'Córdoba' 'Norte de Santander' 'Boyacá' 'Tolima' 'Caquetá' 'La Guajira'
 'Huila' 'Caldas' 'Nariño' 'Putumayo' 'Casanare' 'Risaralda' 'Vichada'
 'Chocó' 'Bogotá D.C.' 'Sucre' 'Amazonas' 'Guaviare'
 'Archipiélago de San Andrés, Providencia y Santa Catalina' 'Guainía'
 'Vaupés']


El dataset trae 38 columnas, pero para este observatorio solo necesitamos 12
(año, sexo, edad, escolaridad, etc.). Seleccionamos esas columnas y les ponemos
nombres más simples y en español.

In [7]:
variables_nombre= [
    'a_o_del_hecho', 'sexo_de_la_victima', 'grupo_de_edad_quinquenal',
    'escolaridad', 'estado_civil',
    'mes_del_hecho', 'dia_del_hecho', 'rango_de_hora_del_hecho_x_3_horas',
    'municipio_del_hecho_dane', 'departamento_del_hecho_dane',
    'circunstancia_del_hecho_detallada', 'medio_de_desplazamiento_o_transporte'
]

df_muertes_nombres = df_muertes[variables_nombre].copy()

df_muertes_variables = df_muertes_nombres.rename(columns={
    'a_o_del_hecho': 'año',
    'sexo_de_la_victima': 'sexo',
    'grupo_de_edad_quinquenal': 'grupo_edad',
    'municipio_del_hecho_dane': 'municipio',
    'departamento_del_hecho_dane': 'departamento',
    'circunstancia_del_hecho_detallada': 'circunstancia',
    'medio_de_desplazamiento_o_transporte': 'medio_desplazamiento'
})

df_muertes_variables

,año,sexo,grupo_edad,escolaridad,estado_civil,mes_del_hecho,dia_del_hecho,rango_de_hora_del_hecho_x_3_horas,municipio,departamento,circunstancia,medio_desplazamiento
0,2015,Hombre,(25 a 29),Básica secundaria,Unión libre,enero,domingo,21:00 a 23:59,San Luis de Cubarral,Meta,Sin información,Motocicleta
1,2015,Hombre,(40 a 44),Básica secundaria,Unión libre,enero,domingo,21:00 a 23:59,San Luis de Cubarral,Meta,Sin información,Motocicleta
2,2015,Hombre,(50 a 54),Preescolar,Unión libre,enero,lunes,00:00 a 02:59,San Luis de Cubarral,Meta,Sin información,Motocicleta
3,2015,Hombre,(25 a 29),Básica secundaria,Unión libre,enero,jueves,09:00 a 11:59,Acacías,Meta,Desobedecer señales de tránsito,Motocicleta
4,2015,Hombre,(35 a 39),Básica secundaria,Unión libre,enero,martes,15:00 a 17:59,San Alberto,Cesar,Otras circunstancias no especificadas,Automóvil
...,...,...,...,...,...,...,...,...,...,...,...,...
73398,2024,Mujer,(70 a 74),Educación básica primaria,Unión libre,Noviembre,Lunes,Sin información,Ibagué,Tolima,Sin información,No aplica
73399,2024,Mujer,(65 a 69),Educación básica primaria,Unión libre,Noviembre,Miércoles,(12:00 a 14:59),Cali,Valle del Cauca,Sin información,No aplica
73400,2024,Hombre,(50 a 54),Educación media o secundaria alta,Unión libre,Noviembre,Lunes,(00:00 a 02:59),Cali,Valle del Cauca,Sin información,Motocicleta
73401,2024,Hombre,(80 y más),Educación básica primaria,Unión libre,Abril,Domingo,Sin información,Cali,Valle del Cauca,Sin información,No aplica


## Sección 2: Lesiones por eventos de transporte

**Fuente:** [Lesiones por eventos de transporte — Colombia](https://www.datos.gov.co/Justicia-y-Derecho/Lesiones-por-eventos-de-transporte-Colombia-a-os-2/ezhf-hscf/about_data)
(Instituto Nacional de Medicina Legal y Ciencias Forenses, vía datos.gov.co)

Misma estructura que la base de muertes, pero cada fila es **una persona lesionada**
(no fallecida) en un evento de transporte. Repetimos el mismo proceso de carga,
limpieza y selección de variables.

In [8]:
url_lesiones = "https://www.datos.gov.co/resource/ezhf-hscf.json?$limit=342796"
respuesta = requests.get(url_lesiones)
datos_lesiones = respuesta.json()

df_lesiones = pd.DataFrame(datos_lesiones)
print(df_lesiones.shape)

(342796, 39)


Revisamos las columnas y los departamentos, igual que hicimos con muertes.

In [9]:
print(df_lesiones.columns)

Index(['id', 'a_o_del_hecho', 'sexo_de_la_victima', 'grupo_de_edad_quinquenal',
       'grupo_mayor_menor_de_edad', 'grupo_de_edad_judicial', 'ciclo_vital',
       'pais_nacimiento', 'escolaridad', 'estado_civil',
       'tipo_de_discapacidad', 'pertenencia_etnica', 'orientacion_sexual',
       'identidad_de_genero', 'transgenero', 'pertenencia_grupal',
       'mes_del_hecho', 'dia_del_hecho', 'rango_de_hora_del_hecho_x_3_horas',
       'codigo_dane_municipio', 'municipio_del_hecho_dane',
       'departamento_del_hecho_dane', 'codigo_dane_departamento',
       'localidad_del_hecho', 'zona_del_hecho', 'escenario_del_hecho',
       'actividad_durante_el_hecho', 'circunstancia_del_hecho_detallada',
       'contexto_del_hecho', 'mecanismo_causal_de_la_lesion_no_fatal',
       'diagnostico_topografico_de_la_lesion_no_fatal',
       'condicion_de_la_victima_at', 'medio_de_desplazamiento_o_transporte',
       'servicio_del_vehiculo', 'clase_o_tipo_de_accidente',
       'objeto_de_colision', '

In [10]:
print(df_lesiones["departamento_del_hecho_dane"].unique())

['Archipiélago de San Andrés, Providencia y Santa Catalina' 'Nariño'
 'Atlántico' 'Boyacá' 'Tolima' 'Cesar' 'Norte de Santander' 'Quindio'
 'Huila' 'Santander' 'Bogotá, D.C.' 'Valle del Cauca' 'Antioquia'
 'Risaralda' 'Magdalena' 'La Guajira' 'Córdoba' 'Cundinamarca' 'Bolívar'
 'Cauca' 'Meta' 'Caldas' 'Putumayo' 'Casanare' 'Chocó' 'Sucre' 'Amazonas'
 'Arauca' 'Guainia' 'Caquetá' 'Vichada' 'Guaviare' 'Vaupés' 'Quindío'
 'Guainía' 'Sin información']


Aquí aparecen tres inconsistencias: `"Quindio"` (sin tilde), `"Guainia"` (sin tilde)
y `"Bogotá, D.C."` (con coma). Las corregimos de la misma forma que en muertes.

In [11]:
df_lesiones["departamento_del_hecho_dane"] = df_lesiones[
    "departamento_del_hecho_dane"
].replace({
    "Quindio": "Quindío",
    "Guainia": "Guainía",
    "Bogotá, D.C.": "Bogotá D.C."
})

In [12]:
departamentos_lesiones = df_lesiones[
    df_lesiones["departamento_del_hecho_dane"] != "Sin información"
]["departamento_del_hecho_dane"].unique()

print(len(departamentos_lesiones))
print(sorted(departamentos_lesiones))

33
['Amazonas', 'Antioquia', 'Arauca', 'Archipiélago de San Andrés, Providencia y Santa Catalina', 'Atlántico', 'Bogotá D.C.', 'Bolívar', 'Boyacá', 'Caldas', 'Caquetá', 'Casanare', 'Cauca', 'Cesar', 'Chocó', 'Cundinamarca', 'Córdoba', 'Guainía', 'Guaviare', 'Huila', 'La Guajira', 'Magdalena', 'Meta', 'Nariño', 'Norte de Santander', 'Putumayo', 'Quindío', 'Risaralda', 'Santander', 'Sucre', 'Tolima', 'Valle del Cauca', 'Vaupés', 'Vichada']


Seleccionamos las mismas 12 variables y les aplicamos los mismos nombres en español.

## Sección 3: Unión de muertes y lesiones

`df_muertes_variables` y `df_lesiones_variables` tienen exactamente las mismas 12
columnas, así que podemos **apilarlas** (unirlas por filas) en una sola tabla de
víctimas. Antes de unirlas, agregamos una columna `gravedad_del_hecho` que indica
de cuál de las dos bases proviene cada registro ("Muerte" o "Lesión") — así no se
pierde esa información al mezclarlas. Aprovechamos también este paso para dejar
solo los años del estudio (2021 a 2024).

In [13]:
variables_nombre = [
    'a_o_del_hecho', 'sexo_de_la_victima', 'grupo_de_edad_quinquenal',
    'escolaridad', 'estado_civil',
    'mes_del_hecho', 'dia_del_hecho', 'rango_de_hora_del_hecho_x_3_horas',
    'municipio_del_hecho_dane', 'departamento_del_hecho_dane',
    'circunstancia_del_hecho_detallada', 'medio_de_desplazamiento_o_transporte'
]

nombres_nuevos = {
    'a_o_del_hecho': 'año',
    'sexo_de_la_victima': 'sexo',
    'grupo_de_edad_quinquenal': 'grupo_edad',
    'municipio_del_hecho_dane': 'municipio',
    'departamento_del_hecho_dane': 'departamento',
    'circunstancia_del_hecho_detallada': 'circunstancia',
    'medio_de_desplazamiento_o_transporte': 'medio_desplazamiento'
}

df_muertes_variables = df_muertes[variables_nombre].copy().rename(columns=nombres_nuevos)
df_muertes_variables['gravedad_del_hecho'] = 'Muerte'

df_lesiones_variables = df_lesiones[variables_nombre].copy().rename(columns=nombres_nuevos)
df_lesiones_variables['gravedad_del_hecho'] = 'Lesión'

df_muertes_lesiones = pd.concat([df_muertes_variables, df_lesiones_variables], ignore_index=True)

# Nos quedamos solo con el rango de años del estudio: 2021 a 2025
df_muertes_lesiones['año'] = df_muertes_lesiones['año'].astype(int)
df_muertes_lesiones = df_muertes_lesiones[
    (df_muertes_lesiones['año'] >= 2021) & (df_muertes_lesiones['año'] <= 2025)
].reset_index(drop=True)

print(df_muertes_lesiones.shape)
df_muertes_lesiones

(153835, 13)


,año,sexo,grupo_edad,escolaridad,estado_civil,mes_del_hecho,dia_del_hecho,rango_de_hora_del_hecho_x_3_horas,municipio,departamento,circunstancia,medio_desplazamiento,gravedad_del_hecho
0,2021,Mujer,(80 y más),Sin información,Casado(a),Abril,Domingo,Sin información,Concordia,Antioquia,Sin información,Automóvil,Muerte
1,2021,Hombre,(45 a 49),Sin información,Unión libre,Noviembre,Martes,Sin información,Concordia,Antioquia,Sin información,Motocicleta,Muerte
2,2022,Hombre,(65 a 69),Educación media o secundaria alta,Soltero(a),enero,sábado,00:00 a 02:59,"Bogotá, D.C.",Bogotá D.C.,Sin información,No aplica,Muerte
3,2022,Hombre,(35 a 39),Educación media o secundaria alta,Soltero(a),enero,miércoles,12:00 a 14:59,"Bogotá, D.C.",Bogotá D.C.,Sin información,Motocicleta,Muerte
4,2022,Hombre,(60 a 64),Educación básica primaria,Unión libre,julio,martes,18:00 a 20:59,Medellín,Antioquia,Desobedecer señales de tránsito,No aplica,Muerte
...,...,...,...,...,...,...,...,...,...,...,...,...,...
153830,2024,Hombre,(60 a 64),Educación básica primaria,Unión libre,Noviembre,Martes,Sin información,Yumbo,Valle del Cauca,Adelantar cerrando,Automóvil,Lesión
153831,2024,Hombre,(55 a 59),Educación inicial y educación preescolar,Unión libre,Noviembre,Martes,(12:00 a 14:59),Yumbo,Valle del Cauca,Cruzar sin observar,Automóvil,Lesión
153832,2024,Mujer,(55 a 59),Sin escolaridad,Unión libre,Noviembre,Martes,Sin información,Yumbo,Valle del Cauca,No respetar prelación,No aplica,Lesión
153833,2024,Hombre,(18 a 19),Educación media o secundaria alta,Soltero (a),Noviembre,Martes,(12:00 a 14:59),Yumbo,Valle del Cauca,Adelantar cerrando,Automóvil,Lesión


# Sección 4: Mercado laboral por departamentos (GEIH — DANE)

**Fuente:** [Mercado laboral por departamentos](https://www.dane.gov.co/index.php/estadisticas-por-tema/mercado-laboral/mercado-laboral-por-departamentos)
(Departamento Administrativo Nacional de Estadística, Gran Encuesta Integrada de
Hogares — GEIH)

A diferencia de las dos fuentes anteriores, aquí cada fila **no es una persona**,
sino un indicador de un departamento en un año determinado (ocupación,
desocupación, subocupación y población total). El archivo viene en formato Excel,
con una hoja distinta para el total, hombres y mujeres.

Descargamos el archivo con `requests` (en vez de dejar que pandas lo descargue
directamente) porque el servidor del DANE devuelve un error `401 Unauthorized`
cuando la solicitud no incluye un encabezado de navegador (`User-Agent`).

In [16]:
from io import BytesIO

url_geih = 'https://www.dane.gov.co/files/operaciones/GEIH/anex-GEIHDepartamentos-2025.xls'
headers = {'User-Agent': 'Mozilla/5.0'}

respuesta = requests.get(url_geih, headers=headers)
xls = pd.ExcelFile(BytesIO(respuesta.content))

print(xls.sheet_names)

['Índice', 'Ficha metodológica', 'Departamentos anual', 'Departamentos anual hombres', 'Departamentos anual mujeres', 'Departamento anual Cabeceras', 'Departamento anual CentrosP', 'Ocu ramas anual  Dptos CIIU 4', 'Errores relativos IML', 'Errores relativos poblaciones']


La siguiente función recorre automáticamente todos los bloques de departamento de
una hoja (localizando cada fila "Concepto"), extrae esos 5 indicadores y arma una
tabla larga (una fila por departamento-año), en vez de tener los años como columnas.

In [17]:
def extraer_geih(xls, sheet_name):
    df = pd.read_excel(xls, sheet_name=sheet_name, header=None)
    filas_concepto = df.index[df[0] == 'Concepto']

    registros = []
    for idx in filas_concepto:
        departamento = df.iloc[idx - 1, 0]
        años = df.iloc[idx + 1, 1:20].values
        ocupacion = df.iloc[idx + 4, 1:20].values
        desocupacion = df.iloc[idx + 5, 1:20].values
        subocupacion = df.iloc[idx + 6, 1:20].values
        poblacion_total = df.iloc[idx + 8, 1:20].values

        for año, ocu, des, sub, pob in zip(años, ocupacion, desocupacion, subocupacion, poblacion_total):
            registros.append({
                'departamento': departamento,
                'año': int(año),
                'ocupacion': ocu,
                'desocupacion': des,
                'subocupacion': sub,
                'poblacion_total': pob
            })

    return pd.DataFrame(registros)

df_anual = extraer_geih(xls, 'Departamentos anual')
df_hombres = extraer_geih(xls, 'Departamentos anual hombres')
df_mujeres = extraer_geih(xls, 'Departamentos anual mujeres')

df_anual.shape, df_hombres.shape, df_mujeres.shape

((437, 6), (437, 6), (437, 6))

Aplicamos la función a las tres hojas: total anual, hombres y mujeres.

In [18]:
df_anual = extraer_geih(xls, 'Departamentos anual')
df_hombres = extraer_geih(xls, 'Departamentos anual hombres')
df_mujeres = extraer_geih(xls, 'Departamentos anual mujeres')

df_anual.shape, df_hombres.shape, df_mujeres.shape

((437, 6), (437, 6), (437, 6))

Las tres hojas dan el mismo tamaño (437 filas × 6 columnas = 23 departamentos × 19
años), lo que confirma que se extrajeron los mismos departamentos y años en las
tres. Ahora le ponemos a cada columna un sufijo según el sexo (`_total`, `_hombre`,
`_mujer`) y **cruzamos** las tres tablas por `departamento` y `año` — a diferencia
de la unión de muertes y lesiones, aquí sí se trata de un cruce real, porque cada
tabla aporta columnas distintas para la misma combinación departamento-año.

In [19]:
df_anual = df_anual.rename(columns={
    'ocupacion': 'ocupacion_total_anual',
    'desocupacion': 'desocupacion_total_anual',
    'subocupacion': 'subocupacion_total_anual',
    'poblacion_total': 'poblacion_total_anual'
})

df_hombres = df_hombres.rename(columns={
    'ocupacion': 'ocupacion_hombre',
    'desocupacion': 'desocupacion_hombre',
    'subocupacion': 'subocupacion_hombre',
    'poblacion_total': 'poblacion_total_hombre'
})

df_mujeres = df_mujeres.rename(columns={
    'ocupacion': 'ocupacion_mujer',
    'desocupacion': 'desocupacion_mujer',
    'subocupacion': 'subocupacion_mujer',
    'poblacion_total': 'poblacion_total_mujer'
})

df_geih = df_anual.merge(df_hombres, on=['departamento', 'año']).merge(df_mujeres, on=['departamento', 'año'])

print(df_geih.shape)
df_geih.head()

(437, 14)


,departamento,año,ocupacion_total_anual,desocupacion_total_anual,subocupacion_total_anual,poblacion_total_anual,ocupacion_hombre,desocupacion_hombre,subocupacion_hombre,poblacion_total_hombre,ocupacion_mujer,desocupacion_mujer,subocupacion_mujer,poblacion_total_mujer
0,Antioquia,2007,54.555216,11.441252,8.339966,5592.744,70.609455,9.093886,8.496139,2695.995,40.224179,14.885198,8.110836,2896.749
1,Antioquia,2008,54.215036,12.332221,9.247148,5659.580,70.258586,9.784394,9.482371,2729.277,39.870363,16.066962,8.902345,2930.303
2,Antioquia,2009,56.740568,12.996154,11.970643,5727.758,71.828960,10.417748,11.991323,2762.952,43.229545,16.569053,11.941986,2964.806
3,Antioquia,2010,57.388462,11.992692,12.252507,5797.516,71.791421,9.240721,12.264435,2797.440,44.470938,15.693836,12.236466,3000.076
4,Antioquia,2011,57.820325,10.769564,10.127097,5866.514,72.762201,8.378211,9.558111,2831.902,44.393894,14.072609,10.913005,3034.612


## Sección 5: Cruce final — accidentalidad y mercado laboral

Con `df_muertes_lesiones` (una fila por víctima) y `df_geih` (una fila por
departamento-año), el siguiente paso es cruzarlas usando `departamento` y `año`
como llave, para agregarle a cada víctima el contexto de mercado laboral de su
departamento en el año del hecho.

Antes del cruce, revisamos si los nombres de departamento coinciden exactamente
entre las dos tablas — un cruce solo encuentra la pareja si el texto es idéntico.

In [21]:
departamentos_geih = set(df_geih["departamento"].unique())
departamentos_eventos = set(df_muertes_lesiones["departamento"].unique())

print("Departamentos que están en ambas bases:")
print(sorted(departamentos_geih & departamentos_eventos))

print("\nDepartamentos que están en GEIH pero NO en eventos:")
print(sorted(departamentos_geih - departamentos_eventos))

print("\nDepartamentos que están en eventos pero NO en GEIH:")
print(sorted(departamentos_eventos - departamentos_geih))

Departamentos que están en ambas bases:
['Antioquia', 'Atlántico', 'Bolívar', 'Boyacá', 'Caldas', 'Caquetá', 'Cauca', 'Cesar', 'Chocó', 'Cundinamarca', 'Córdoba', 'Huila', 'La Guajira', 'Magdalena', 'Meta', 'Nariño', 'Norte de Santander', 'Quindío', 'Risaralda', 'Santander', 'Sucre', 'Tolima', 'Valle del Cauca']

Departamentos que están en GEIH pero NO en eventos:
[]

Departamentos que están en eventos pero NO en GEIH:
['Amazonas', 'Arauca', 'Archipiélago de San Andrés, Providencia y Santa Catalina', 'Bogotá D.C.', 'Casanare', 'Guainía', 'Guaviare', 'Putumayo', 'Sin información', 'Vaupés', 'Vichada']


## Sección 6: Construcción de la base de datos maestra

`df_muertes_lesiones` ya está restringida al rango de años del estudio (2021-2025).
Ahora, con el diagnóstico de departamentos de la sección anterior, nos quedamos
únicamente con los 23 departamentos que aparecen en la base del DANE (GEIH),
omitiendo cualquier registro de víctimas de un departamento que no esté en esa
lista, y cruzamos ambas tablas usando `departamento` y `año` como llave.

De todos los indicadores de la GEIH, en la base maestra final solo conservamos
cuatro variables: la tasa de ocupación, desocupación y subocupación según el sexo
de la víctima (`ocupacion_segun_sexo`, `desocupacion_segun_sexo`,
`subocupacion_segun_sexo`), y la tasa de desocupación total del departamento
(`desocupacion_total_anual`). El resto de indicadores de la GEIH (ocupación y
subocupación totales, población total, hombre y mujer) no se incluyen en la
base final.

In [22]:
import numpy as np

# Nos quedamos solo con los 23 departamentos del DANE (GEIH); se omite
# cualquier registro cuyo departamento no esté en esa lista
df_eventos_filtrado = df_muertes_lesiones[
    df_muertes_lesiones['departamento'].isin(departamentos_geih)
].copy()

df_maestra = df_eventos_filtrado.merge(
    df_geih,
    on=['departamento', 'año'],
    how='left'
)

# Asignamos la tasa de ocupacion/desocupacion/subocupacion segun el sexo
# de la victima, para no tener columnas separadas de hombre y mujer
sexo_normalizado = df_maestra['sexo'].astype(str).str.strip().str.lower()
condiciones = [sexo_normalizado == 'hombre', sexo_normalizado == 'mujer']

df_maestra['ocupacion_segun_sexo'] = np.select(
    condiciones, [df_maestra['ocupacion_hombre'], df_maestra['ocupacion_mujer']], default=np.nan
)
df_maestra['desocupacion_segun_sexo'] = np.select(
    condiciones, [df_maestra['desocupacion_hombre'], df_maestra['desocupacion_mujer']], default=np.nan
)
df_maestra['subocupacion_segun_sexo'] = np.select(
    condiciones, [df_maestra['subocupacion_hombre'], df_maestra['subocupacion_mujer']], default=np.nan
)

# De la GEIH, en la base maestra solo dejamos estas 4 variables:
# ocupacion_segun_sexo, desocupacion_segun_sexo, subocupacion_segun_sexo y desocupacion_total_anual
df_maestra = df_maestra.drop(columns=[
    'ocupacion_hombre', 'ocupacion_mujer',
    'desocupacion_hombre', 'desocupacion_mujer',
    'subocupacion_hombre', 'subocupacion_mujer',
    'ocupacion_total_anual', 'subocupacion_total_anual',
    'poblacion_total_anual', 'poblacion_total_hombre', 'poblacion_total_mujer'
])

print(df_maestra.shape)
print(df_maestra['departamento'].nunique(), "departamentos en la base maestra")
df_maestra.head()

(122874, 17)
23 departamentos en la base maestra


,año,sexo,grupo_edad,escolaridad,estado_civil,mes_del_hecho,dia_del_hecho,rango_de_hora_del_hecho_x_3_horas,municipio,departamento,circunstancia,medio_desplazamiento,gravedad_del_hecho,desocupacion_total_anual,ocupacion_segun_sexo,desocupacion_segun_sexo,subocupacion_segun_sexo
0,2021,Mujer,(80 y más),Sin información,Casado(a),Abril,Domingo,Sin información,Concordia,Antioquia,Sin información,Automóvil,Muerte,13.691361,36.684528,17.161732,5.926587
1,2021,Hombre,(45 a 49),Sin información,Unión libre,Noviembre,Martes,Sin información,Concordia,Antioquia,Sin información,Motocicleta,Muerte,13.691361,66.433591,11.437208,5.315065
2,2022,Hombre,(60 a 64),Educación básica primaria,Unión libre,julio,martes,18:00 a 20:59,Medellín,Antioquia,Desobedecer señales de tránsito,No aplica,Muerte,10.057339,71.016665,7.919409,5.100190
3,2022,Hombre,(55 a 59),Sin información,Soltero(a),julio,martes,18:00 a 20:59,Medellín,Antioquia,Desobedecer señales de tránsito,No aplica,Muerte,10.057339,71.016665,7.919409,5.100190
4,2022,Mujer,(20 a 24),Educación técnica profesional y tecnológica,Unión libre,noviembre,lunes,09:00 a 11:59,Planeta Rica,Córdoba,Sin información,Buseta,Muerte,12.476432,38.617998,17.207031,8.364303


Exportamos la base de datos maestra final.

In [24]:
df_maestra.to_csv('base_maestra_accidentalidad_colombia.csv', index=False, encoding='utf-8-sig')
print("Archivo exportado correctamente.")

Archivo exportado correctamente.


## **ENLACE DIRECTO A LA BASE DE DATOS CREADA**  ⬇️


https://drive.google.com/file/d/1scXzoYoH9dImZJhG4sTJjZ6BrST2TvKb/view?usp=drive_link

## **Referencias**

Agencia Nacional de Seguridad Vial. (s. f.). *Observatorio Nacional de Seguridad
Vial*. https://ansv.gov.co/observatorio/que_es_observatorio

Departamento Administrativo Nacional de Estadística. (2025). *Mercado laboral por
departamentos* [Boletín técnico]. https://www.dane.gov.co/index.php/estadisticas-por-tema/mercado-laboral/mercado-laboral-por-departamentos